# ARE Benchmark — Modelos Locais via Ollama

Replica o benchmark [ARE (An R Eval)](https://github.com/diegoamrg4123/are-dataset-csv) usando modelos locais OSS via Ollama.  
O juiz é também um modelo local — sem dependência de APIs externas.

**Dependências:** `pip install ollama pandas matplotlib seaborn`  
**Pré-requisito:** Ollama rodando (`ollama serve`) com os modelos instalados.

In [ ]:
import pandas as pd
import ollama
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
from datetime import datetime
from IPython.display import display

sns.set_theme(style="whitegrid", palette="muted")

## Configuração
Edite `MODELS` e `JUDGE_MODEL` conforme os modelos instalados (`ollama list`).

In [ ]:
OLLAMA_HOST = "http://192.168.1.37:11434"

MODELS = [
    "lfm2.5-thinking:1.2b",
    "qwen3.5:2b",
    "qwen2.5-coder:3b",
    "gemma4:e2b",
]

JUDGE_MODEL = "gpt-oss:20b"

EPOCHS = 3

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

client = ollama.Client(host=OLLAMA_HOST)
print(f"Conectado ao Ollama em {OLLAMA_HOST}")

## Dataset

In [3]:
url = "https://raw.githubusercontent.com/diegoamrg4123/are-dataset-csv/main/are_dataset.csv"
df = pd.read_csv(url, sep=";", encoding="utf-8")
print(f"{len(df)} problemas carregados")
display(df.head(2))

29 problemas carregados


,id,input,target,domain,task,source,knowledge
0,after-stat-bar-heights,This bar chart shows the count of different cu...,Preferably: \n\n```\nggplot(data = diamonds) +...,Data analysis,New code,https://jrnold.github.io/r4ds-exercise-solutio...,tidyverse
1,conditional-grouped-summary,I have a set of data not unlike the following ...,One solution is to `group_by()` and summarize:...,Data analysis,New code,https://forum.posit.co/t/dplyr-case-when-summa...,tidyverse


## Solver
Envia o enunciado para o modelo avaliado e retorna a resposta.

In [ ]:
SOLVER_SYSTEM = "You are an expert R programmer. Answer concisely with working R code."

def solve(model: str, problem: str) -> str:
    response = client.chat(
        model=model,
        messages=[
            {"role": "system", "content": SOLVER_SYSTEM},
            {"role": "user",   "content": problem}
        ]
    )
    return response["message"]["content"]

## Juiz / Scorer (partial credit)
Replica `model_graded_qa(partial_credit=TRUE)` do script R original.  
O juiz classifica cada resposta como **C** (correto), **P** (parcial) ou **I** (incorreto).

In [ ]:
JUDGE_PROMPT = """\
You are an expert R programmer evaluating an answer to an R coding task.

## Question
{question}

## Expected answer / grading rubric
{target}

## Model response
{response}

Grade the response with a single letter:
- C  (correct)  — fully achieves the task
- P  (partial)  — on the right track but incomplete or has minor errors
- I  (incorrect) — wrong or completely off

Reply with ONLY the letter C, P, or I."""

SCORE_MAP = {"C": 1.0, "P": 0.5, "I": 0.0}

def judge(question: str, target: str, response: str) -> float:
    result = client.chat(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(
            question=question, target=target, response=response
        )}]
    )
    letter = result["message"]["content"].strip().upper()[0]
    return SCORE_MAP.get(letter, 0.0)

## Runner
Itera sobre todos os problemas × epochs para um modelo. Pula se o resultado já existir.

In [6]:
def run_benchmark(model: str, overwrite: bool = False):
    safe_name = model.replace(":", "_").replace("/", "_")
    result_path = RESULTS_DIR / f"{safe_name}.json"

    if not overwrite and result_path.exists():
        print(f"Skipping {model} — já existe em {result_path}")
        return

    records = []
    total = len(df) * EPOCHS
    done = 0

    for epoch in range(1, EPOCHS + 1):
        for _, row in df.iterrows():
            done += 1
            print(f"[{model}] epoch {epoch}/{EPOCHS} | {row['id']} ({done}/{total})", end=" ... ")
            response = solve(model, row["input"])
            score    = judge(row["input"], row["target"], response)
            records.append({
                "model":    model,
                "epoch":    epoch,
                "id":       row["id"],
                "domain":   row["domain"],
                "task":     row["task"],
                "score":    score,
                "response": response,
                "ts":       datetime.now().isoformat()
            })
            print(f"score={score}")

    result_path.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
    mean_score = sum(r["score"] for r in records) / len(records)
    print(f"\n{model} — mean score: {mean_score:.3f} | salvo em {result_path}\n")

## Execução
Roda o benchmark para todos os modelos em `MODELS`. Modelos já avaliados são pulados automaticamente.

In [ ]:
for model in MODELS:
    run_benchmark(model)

[lfm2.5-thinking:1.2b] epoch 1/3 | after-stat-bar-heights (1/87) ... score=0.0
[lfm2.5-thinking:1.2b] epoch 1/3 | conditional-grouped-summary (2/87) ... score=0.0
[lfm2.5-thinking:1.2b] epoch 1/3 | correlated-delays-reasoning (3/87) ... score=0.0
[lfm2.5-thinking:1.2b] epoch 1/3 | curl-http-get (4/87) ... score=1.0
[lfm2.5-thinking:1.2b] epoch 1/3 | dropped-level-legend (5/87) ... score=0.5
[lfm2.5-thinking:1.2b] epoch 1/3 | filter-multiple-conditions (6/87) ... 

## Análise de Resultados

In [ ]:
all_records = []
for p in RESULTS_DIR.glob("*.json"):
    all_records.extend(json.loads(p.read_text(encoding="utf-8")))

if not all_records:
    print("Nenhum resultado encontrado em results/. Execute o benchmark primeiro.")
else:
    results_df = pd.DataFrame(all_records)

    print("=== Score médio por modelo ===")
    summary = (
        results_df
        .groupby("model")["score"]
        .agg(mean="mean", std="std", n="count")
        .sort_values("mean", ascending=False)
        .round(3)
    )
    display(summary)

In [ ]:
if all_records:
    print("=== Score médio por modelo × domínio ===")
    domain_summary = (
        results_df
        .groupby(["model", "domain"])["score"]
        .mean()
        .unstack()
        .round(3)
    )
    display(domain_summary)

    print("\n=== Score médio por modelo × tipo de tarefa ===")
    task_summary = (
        results_df
        .groupby(["model", "task"])["score"]
        .mean()
        .unstack()
        .round(3)
    )
    display(task_summary)

## Visualizações

In [ ]:
if not all_records:
    print("Nenhum resultado encontrado em results/. Execute o benchmark primeiro.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    fig.suptitle("ARE Benchmark — Resultados por Modelo", fontsize=14, fontweight="bold")

    # 1. Score médio por modelo (barras horizontais com desvio padrão)
    ax1 = axes[0, 0]
    s = summary.sort_values("mean")
    colors = sns.color_palette("muted", len(s))
    ax1.barh(s.index, s["mean"], xerr=s["std"], color=colors, capsize=4, height=0.6)
    ax1.set_xlabel("Score médio")
    ax1.set_title("Score médio por modelo (± std)")
    ax1.set_xlim(0, 1)
    for i, (val, err) in enumerate(zip(s["mean"], s["std"])):
        ax1.text(min(val + err + 0.02, 0.97), i, f"{val:.3f}", va="center", fontsize=9)

    # 2. Heatmap modelo × domínio
    ax2 = axes[0, 1]
    sns.heatmap(
        domain_summary, annot=True, fmt=".2f", cmap="YlGn",
        vmin=0, vmax=1, linewidths=0.5, ax=ax2, cbar_kws={"shrink": 0.8}
    )
    ax2.set_title("Score por domínio")
    ax2.set_ylabel("")
    ax2.tick_params(axis="x", rotation=30)

    # 3. Heatmap modelo × tipo de tarefa
    ax3 = axes[1, 0]
    sns.heatmap(
        task_summary, annot=True, fmt=".2f", cmap="YlGn",
        vmin=0, vmax=1, linewidths=0.5, ax=ax3, cbar_kws={"shrink": 0.8}
    )
    ax3.set_title("Score por tipo de tarefa")
    ax3.set_ylabel("")
    ax3.tick_params(axis="x", rotation=30)

    # 4. Proporção C / P / I por modelo (stacked bar)
    ax4 = axes[1, 1]
    grade_map = {1.0: "Correto", 0.5: "Parcial", 0.0: "Incorreto"}
    results_df["grade"] = results_df["score"].map(grade_map)
    grade_pct = (
        results_df.groupby(["model", "grade"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["Correto", "Parcial", "Incorreto"])
        .apply(lambda x: x / x.sum(), axis=1)
    )
    grade_pct.plot(
        kind="barh", stacked=True, ax=ax4,
        color=["#4caf50", "#ff9800", "#f44336"], width=0.6
    )
    ax4.set_xlabel("Proporção")
    ax4.set_title("Distribuição Correto / Parcial / Incorreto")
    ax4.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax4.legend(loc="lower right", fontsize=9)

    plt.tight_layout()
    out_path = RESULTS_DIR / "benchmark_results.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Gráfico salvo em {out_path}")